# Create YOLO dataset

## Introduction

This guide provides a structured, code-first approach to preparing and validating a dataset for YOLO model training inside the Hibou project.
Here, you’ll find modular scripts and functions for:

- Data acquisition and conversion (downloading, format conversion to YOLO)
- Dataset validation (path checks, duplicate detection, orphaned files, and visual confirmation)
- Optional utilities (label conversion, class ID management, dataset splitting/merging, and file renaming)

## Table of Contents


<!-- TOC -->
* [🏁 1. Initialization](#-1-initialization)
    * [1.0 Installing dependencies](#10-installing-dependencies)
    * [1.1 Importing Libraries](#11-importing-libraries)
    * [1.2 Global Definitions](#12-global-definitions)
    * [1.3 Global Settings](#13-global-settings)
    * [1.4 Global Structure](#14-global-structure)
    * [1.5 Global Function Definitions](#15-global-function-definitions)
* [📂 2. Dataset](#-2-dataset)
    * [2.1 Download dataset](#21-download-dataset)
    * [2.2 Convert to Yolo](#22-convert-to-yolo)
    * [2.3 Dataset Check](#23-dataset-check)
      * [Directory path validation](#directory-path-validation)
      * [Duplicate files](#duplicate-files)
        * [⚠️Delete duplicate files ⚠️](#delete-duplicate-files-)
      * [Check for non-valid txt files](#check-for-non-valid-txt-files)
      * [Check for orphelins txt file](#check-for-orphelins-txt-file)
        * [⚠️Delete orphelin labels files ⚠️](#delete-orphelin-labels-files-)
      * [Check for orphelins image file](#check-for-orphelins-image-file)
        * [⚠️Delete orphelin labels files ⚠️](#delete-orphelin-labels-files--1)
    * [2.3 Visual Confirmation](#23-visual-confirmation)
    * [2.4 Upload](#24-upload)
* [⚙️ 3. Tools (Optional)](#-3-tools-optional)
  * [🧾 3.1 Labels](#-31-labels)
    * [3.1.1 Convert label format](#311-convert-label-format)
      * [From COCO format to YOLO](#from-coco-format-to-yolo)
    * [3.1.2 Auto repare txt labels](#312-auto-repare-txt-labels)
    * [3.1.3 Change class ID](#313-change-class-id)
    * [3.1.4 Set labels (SAM 3)](#314-set-labels)
        * [Load model](#load-model)
        * [Settings](#settings)
  * [🗃️  3.2 Files](#-32-files)
    * [3.1.2 Split dataset into folders](#312-split-dataset-into-folders)
    * [3.1.3 Merge file from split folders](#313-merge-file-from-split-folders)
    * [3.1.4  Rename files](#314--rename-files)
<!-- TOC -->

# 🏁 1. Initialization

### 1.0 Installing dependencies

In [ ]:
!pip install torch==2.7.0 torchvision torchaudio --index-url https://download.pytorch.org/whl/cu126 -q
!pip install sam3 scikit-learn requests ipywidgets python-dotenv datasets opencv-python -q

### 1.1 Importing Libraries

In [ ]:
from datasets import Dataset, DatasetDict, load_dataset, Image, Features, Value
from collections import defaultdict
from typing import Iterable, Union
from huggingface_hub import login
from PIL import Image as PILImage
from PIL import Image as PILImage
import matplotlib.pyplot as plt
from dotenv import load_dotenv
from PIL import ImageDraw
from pathlib import Path

import datetime
import hashlib
import random
import shutil
import math
import cv2
import ast
import re
import os

### 1.2 Global Definitions

- **DATASET_ROOT_DIR**: Path of the dataset, directory could be empty or contain images and labels.
- **DATASET_SPLIT_NAME**: Name of the split for Hugging Face. For dataset creation it should have only one split.
- **MODELS_DIRECTORY**: Directory where models are stored.

In [ ]:
DATASET_ROOT_DIR = Path('../datasets/work')
DATASET_SPLIT_NAME = DATASET_ROOT_DIR / 'train_validation_test'
MODELS_DIRECTORY = Path('./models/')
ALLOWED_EXTENSIONS = [".jpg", ".jpeg", ".JPG", ".JPEG", ".mp4"]
load_dotenv()

### 1.3 Global Settings

Log in to the different providers.

- **Hugging Face**: To download and upload dataset

In [ ]:
login()  # Keep commented if token loaded from .env file

### 1.4 Global Structure

`dataset_structure` is used to simplify files process through the differents steps. It will be filled automatically later. You don't need to change anything.

In [ ]:
dataset_structure = {
    "root": Path(""),
    "name": "",
    "classes": [],
    "images": [],
    "labels": [],
    "class_counts": {}
}

### 1.5 Global Function Definitions

In [ ]:
def list_files(
        directory: Union[str, Path],
        extensions: Iterable[str],
        include_root_directory: bool = False,
        recursive: bool = False,
) -> list[Path]:
    directory = Path(directory)
    extensions = tuple(extensions)

    matched_files = []

    if recursive:
        iterator = directory.rglob("*")
    else:
        iterator = directory.iterdir()

    for p in iterator:
        if p.is_file() and p.suffix in extensions:
            matched_files.append(
                p if include_root_directory else p.name
            )

    return matched_files


def numeric_key(name):
    """Extract the first number from a filename for sorting."""
    nums = re.findall(r'\d+', name)
    return int(nums[0]) if nums else float('inf')


def sort_files_by_number(files: list):
    """
    Sort a list of filenames by the first number found in each name.

    Args:
        files (list): List of filenames (strings)

    Returns:
        list: Sorted list of filenames
    """
    return sorted(files, key=lambda i: int(i.stem))


def parse_label(data: Union[Path, str]):
    try:
        content = data.read_text() if isinstance(data, Path) else data
        return [x for x in re.split(r"\s+", content) if x]
    except Exception as e:
        print(f"Error parsing label {data}: {e}")
        return []

def update_dataset_structure():
    dataset_structure["root"] = Path(DATASET_ROOT_DIR)
    dataset_structure["name"] = DATASET_ROOT_DIR.name
    dataset_structure["classes"] = ["drone", "other"]

    dataset_structure["images"] = sort_files_by_number(
        list_files(DATASET_SPLIT_NAME, ALLOWED_EXTENSIONS, True)
    )
    dataset_structure["labels"] = sort_files_by_number(
        list_files(DATASET_SPLIT_NAME, [".txt"], True)
    )

    for class_name in dataset_structure["classes"]:
        dataset_structure["class_counts"][class_name] = 0

    for label in dataset_structure["labels"]:
        parsed_label = parse_label(label)
        for i in range(0, len(parsed_label), 5):
            try:
                dataset_structure["class_counts"][dataset_structure["classes"][int(parsed_label[i])]] += 1
            except Exception as e:
                print(f"Error parsing label {label}: {e}")


def delete_files_in_dataset(files_to_delete: list):
    try:
        confirm = input("Files are going to be deleted. Type 'yes' to continue: ").strip().lower()
        if confirm != 'yes':
            print("Deletion aborted by user.")
            return

        for file in files_to_delete:
            if os.path.isfile(file):
                os.remove(file)
                print(f"Deleted: {file}")
            else:
                print(f"Warning: File does not exist: {file}")

    except KeyboardInterrupt:
        print("\nDeletion aborted by user (KeyboardInterrupt).")


def backup_dataset():
    dataset_path = dataset_structure.get("path", "")
    backup_dir = os.path.join("./datasets", ".backup")
    os.makedirs(backup_dir, exist_ok=True)

    now = datetime.datetime.now().strftime('%Y-%m-%d_%H-%M-%S')
    target_name = f"{dataset_structure["name"]}-{now}"
    target_path = os.path.join(backup_dir, target_name)

    # Copytree with ignoring to exclude the backup folder itself
    def ignore_backup(_, names):
        return {"backup"} if "backup" in names else set()

    shutil.copytree(dataset_path, target_path, ignore=ignore_backup)
    print(f"Backup created at: {target_path}")
    return str(target_path)


def load_image_with_boxes(image_path, label_path):
    """Returns a PIL image with YOLO bounding boxes drawn."""
    img = PILImage.open(image_path)
    draw = ImageDraw.Draw(img)

    colors = ["red", "blue", "green", "orange"]

    w, h = img.size

    if os.path.exists(label_path):
        with open(label_path, "r") as f:
            lines = f.readlines()

        for line in lines:
            if line == '1':
                continue
            cls, xc, yc, bw, bh = map(float, line.split())

            color = colors[int(cls)]

            # YOLO normalized → pixel coordinates
            x_center = xc * w
            y_center = yc * h
            box_width = bw * w
            box_height = bh * h

            x1 = x_center - box_width / 2
            y1 = y_center - box_height / 2
            x2 = x_center + box_width / 2
            y2 = y_center + box_height / 2

            draw.rectangle([x1, y1, x2, y2], outline=color, width=3)
            draw.text((x1, y1), f"{int(cls)}", fill="red")

    return img

# 📂 2. Dataset

### 2.1 Download dataset

Download the dataset from Hugging Face. [Hibou-Foundation](https://huggingface.co/Hibou-Foundation) is the official repo of the project.

If the dataset is already downloaded and ready or you want to prepare a custom dataset, you can skip this step and go directly to step: **2.3 Dataset Check**


In [ ]:
dataset = load_dataset("Hibou-Foundation/computer-vision")

### 2.2 Convert to Yolo

Once the dataset is downloaded, it must be converted back into images and labels.

In [ ]:
# Create folders
for split in ["train_validation_test"]:
    os.makedirs(f"{DATASET_ROOT_DIR}/{split}", exist_ok=True)


def export_to_yolo(ds, split_name):
    for idx, sample in enumerate(ds):
        try:
            image = sample["image"]  # already a PIL.Image
            label = sample["raw_label"]  # already YOLO format [[class, cx, cy, w, h], ...]

            # Save image
            img_path = f"{DATASET_ROOT_DIR}/{split_name}/{idx}.jpg"
            image.save(img_path, quality=95)

            # Save labels
            lbl_path = f"{DATASET_ROOT_DIR}/{split_name}/{idx}.txt"
            with open(lbl_path, "w") as f:
                f.write(label)
        except Exception as e:
            print(f"Error processing sample {idx}: {e}")
            continue


# Run export
split_mapping = {"train_validation_test": "train_validation_test"}
for hf_split, folder_name in split_mapping.items():
    export_to_yolo(dataset[hf_split], folder_name)

### 2.3 Dataset Check

This section aims to check the integrity of the dataset. It will cover the following steps:

- Duplicate files
- Check for non-valid txt files
- Check for orphelins txt file
- Check for orphelins image file

If you need to do some operations on the dataset, please refer to the section: **3. Tools**

#### Directory path validation

Check if the paths defined in the section: **1.2 Global Definitions** are correct.

In [ ]:
update_dataset_structure()

print(f"Dataset path: {DATASET_SPLIT_NAME}")
print(f"Dataset name: {dataset_structure['name']}\n")

print(f"Number of images : {len(dataset_structure['images'])}")
print(f"Number of labels : {len(dataset_structure['labels'])}\n")

for class_name in dataset_structure["classes"]:
    instance_count = dataset_structure["class_counts"][class_name]
    total_instance_count = sum(dataset_structure["class_counts"].values())
    if total_instance_count != 0:
        ratio = instance_count * 100 / total_instance_count
    else:
        ratio = 0

    print(
        f"Class: {class_name:<10} | Instances: {instance_count:>6} | Ratio: {ratio:>6.2f}%"
    )

#### Check for non RGBA images

In [ ]:
image_paths = dataset_structure["images"]
non_rgb_images = []

for p in image_paths:
    try:
        with PILImage.open(p) as img:
            if img.mode != "RGB":
                print(f"{p} -> {img.mode}")
                non_rgb_images.append(p)

    except Exception as e:
        print(f"Error reading {p}: {e}")

print(f"Done.")

##### ⚠️Delete non-RGB images ⚠️

In [ ]:
delete_files_in_dataset(non_rgb_images)
update_dataset_structure()

#### Duplicate files

Check for duplicate images and labels.

In [ ]:
duplicate_paths_to_check = [
    dataset_structure["images"],
]


def sha3_file(path, chunk_size=8192):
    hash_sha3 = hashlib.sha3_256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(chunk_size), b""):
            hash_sha3.update(chunk)
    return hash_sha3.hexdigest()


duplicates_array = []

for path in duplicate_paths_to_check:

    # Build list of (hash, path)
    pairs = [
        (sha3_file(f), f)
        for f in path
    ]

    # Group paths by hash
    hash_map = defaultdict(list)
    for h, p in pairs:
        hash_map[h].append(p)

    # Extract only duplicates
    duplicates = {h: paths for h, paths in hash_map.items() if len(paths) > 1}
    duplicates_array.append(duplicates)

    # Print results
    if duplicates:
        print("Duplicate files found:")
        for h, paths in duplicates.items():
            print(f"Hash: {h}")
            for p in paths:
                print(f"  - {p}")
    else:
        print(f"No duplicate files found in the scanned directories.")

##### ⚠️Delete duplicate files ⚠️

Delete duplicate images and labels.

In [ ]:
try:
    if input("Files are going to be deleted. Write yes to continue.") == 'yes':
        for duplicates in duplicates_array:
            for paths in duplicates.values():
                os.remove(paths[0])

except KeyboardInterrupt:
    print("Deletion aborted.")
finally:
    update_dataset_structure()

#### Check for non-valid txt files

This section will check if:

- The file contains at least one label
- The labels contain the right number of elements
- Contains only numerical values

If the check fails, for the following reasons:

_Label file is not divisible by 5 elements_: And the printed array looks like: `["0", "0.23", "0.45", "0.67", "0.89", ""]` The error is caused by `""` at the end of the array. To quickly resolve the issue, please refer to the section: **3.1.2 Auto repare txt labels**

In [ ]:
empty_label_paths_to_check = [
    dataset_structure["labels"]
]

empty_label = []

for path_list in empty_label_paths_to_check:
    for p in path_list:

        with open(p, "r") as f:
            content = f.read()
        content = re.split(r"\s+", content)
        content_length = len(content)

        if content_length == 1 and content[0] == '1':
            # Allow check bypass, non-drone class
            continue

        if content_length <= 4:
            empty_label.append([p, "Not enough elements in the label file."])
            continue

        if content_length % 5 != 0:
            empty_label.append([p, "Label file is not divisible by 5 elements."])
            continue

        for i in range(0, content_length, 5):
            if not content[i].isdigit():
                empty_label.append([p, "Label file contains non-numeric elements."])
                continue

for e in empty_label:
    print(f"{e[0]}: {e[1]}")

print("Done")

#### Check for orphelins txt file

In [ ]:
orphans_duplicate_paths_to_check = [
    dataset_structure["labels"],
]

orphelins_label = []

for path_list in orphans_duplicate_paths_to_check:
    for p in path_list:

        root, _ = os.path.splitext(p)

        candidates = [
            root + ".jpg",
            root + ".JPG",
            root + ".jpeg",
            root + ".JPEG",
            root + ".png",
            root + ".PNG",
        ]

        if not any(os.path.isfile(c) for c in candidates):
            orphelins_label.append(p)
            print(f"TXT file {p} does not belong to any image.")
print("Done")

##### ⚠️Delete orphelin labels files ⚠️

Delete labels that don't belong to any images.

In [ ]:
delete_files_in_dataset(orphelins_label)
update_dataset_structure()

#### Check for orphelins image file

In [ ]:
orphans_images_paths_to_check = [
    dataset_structure["images"],
]

orphelins_images = []

for path_list in orphans_images_paths_to_check:
    for p in path_list:

        root, _ = os.path.splitext(p)

        candidates = [
            root + ".txt",
        ]

        if not any(os.path.isfile(c) for c in candidates):
            orphelins_images.append(p)
            print(f"TXT file {p} does not belong to any label.")
print("Done")

##### ⚠️Delete orphelin labels files ⚠️

Delete images that don't belong to any labels.

In [ ]:
delete_files_in_dataset(orphelins_images)
update_dataset_structure()

### 2.3 Visual Confirmation

Load N random images and show rectangles around the differents classes.

In [ ]:
N = 200  # number of random samples
cols = 4

%matplotlib inline

# list all images
image_paths = [p for p in os.listdir(DATASET_SPLIT_NAME) if p.lower().endswith((".jpg", ".png"))]
sampled_images = random.sample(image_paths, min(N, len(image_paths)))

# -------- DISPLAY GRID -------- #

rows = math.ceil(len(sampled_images) / cols)

fig, axes = plt.subplots(rows, cols, figsize=(cols * 3, rows * 2))
axes = axes.flatten()

for ax, img_name in zip(axes, sampled_images):
    img_path = os.path.join(DATASET_SPLIT_NAME, img_name)
    label_path = os.path.join(DATASET_SPLIT_NAME, img_name.rsplit(".", 1)[0] + ".txt")

    img = load_image_with_boxes(img_path, label_path)

    ax.imshow(img)
    ax.axis("off")
    ax.set_title(img_name, fontsize=8)

# turn off unused axes
for ax in axes[len(sampled_images):]:
    ax.axis("off")

plt.tight_layout()
plt.show()

### 2.4 Upload

Prepare the dataset to be uploaded on Hugging Face.
There is nothing to do here.

In [ ]:
update_dataset_structure()


def load_split(image_dir: Path, label_dir: Path):
    data = []
    for img_path in image_dir.glob("*.jpg"):
        label_path = label_dir / f"{img_path.stem}.txt"
        with open(label_path) as f:
            label = str(f.read().strip())  # parse label as needed
        class_id = int(label.split()[0])
        class_name = dataset_structure["classes"][class_id]
        if len(label.split()) > 1:
            box = label.split()[1:]
        else:
            box = None

        data.append(
            {
                "image": str(img_path),
                "class_id": class_id,
                "class_name": class_name,
                "box": box,
                "name": img_path.name,
                "raw_label": label,
            }
        )

    features = Features({
        "image": Image(),
        "class_id": Value("int64"),
        "class_name": Value("string"),
        "box": [Value("string")],
        "name": Value("string"),
        "raw_label": Value("string"),
    })

    return Dataset.from_list(data, features=features)


# Paths
train_image_dir = Path(DATASET_SPLIT_NAME)
train_label_dir = Path(DATASET_SPLIT_NAME)

# Load datasets
train_dataset = load_split(train_image_dir, train_label_dir)

# Combine into DatasetDict
dataset = DatasetDict({
    "train_validation_test": train_dataset,
})

dataset

In [ ]:
commit_message: str | None = None
commit_description: str | None = None
dataset.push_to_hub(
    repo_id="Hibou-Foundation/computer-vision",
    commit_message=commit_message,
    commit_description=commit_description
)

# ⚙️ 3. Tools (Optional)

## 🧾 3.1 Labels

Functions declerations

In [ ]:
def normalize_to_yolo(bbox_xywh: list, img_w, img_h):
    assert len(bbox_xywh) == 4, "bbox_xywh must have four elements"
    assert all(isinstance(x, (int, float)) for x in bbox_xywh), "bbox_xywh must be a list of numbers"
    x, y, w, h = bbox_xywh

    x_center = x + w / 2
    y_center = y + h / 2

    return [
        x_center / img_w,
        y_center / img_h,
        w / img_w,
        h / img_h,
    ]


def write_yolo_label(label_path: Path, labels: list, normalize: bool = False, img_w=640, img_h=480):
    assert label_path.suffix == ".txt", "Path must end with .txt"
    lines_to_write = []
    for label in labels:
        assert len(label) == 5, "Each label must have five elements"
        cords = normalize_to_yolo(label[1:5], img_w=img_w, img_h=img_h) if normalize else label[1:5]
        class_id = label[0]
        lines_to_write.append(f"{class_id} {cords[0]:.6f} {cords[1]:.6f} {cords[2]:.6f} {cords[3]:.6f}")
    with open(label_path, "w") as f:
        f.write("\n".join(lines_to_write))

### 3.1.1 Convert label format

This section aims to help to convert différents labels format to Yolo's format.

#### From COCO format to YOLO

Convert COCO label to Yolo with the following COCO structure.

```json
{
 'width': 640,
 'height': 480,
 'objects': {
    'bbox': [[281.0, 210.0, 25.0, 19.0]],  // COCO format: [x, y, width, height]
    'category': [0],  // Category index for the drone
    'area': [475.0],  // Area of the bounding box
    'id': [0]        // Object ID
 },
 'image': <PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=640x480>,
 'image_id': 2
}
```

In [ ]:
labels_directory = Path(DATASET_SPLIT_NAME)

labels_path = list_files(labels_directory, extensions=['.txt'], include_root_directory=True)
for path in labels_path:
    with open(path, 'r') as f:
        content = f.read()
    data = ast.literal_eval(content)
    bbox = data.get('bbox')
    category = data.get('category')
    yolo_lines = []
    for box, id in zip(bbox, category):
        yolo_lines.append([id, box[0], box[1], box[2], box[3]])
    write_yolo_label(path, yolo_lines, normalize=True, img_w=640, img_h=480)
print("Done")

### 3.1.2 Auto repare txt labels

Repaire sigthly damaged labels.

- Remove the "" at the end of each line.
- Add missing break line between two labels.

In [ ]:
labels_to_repaire = [
    dataset_structure["labels"]
]

for path_list in labels_to_repaire:
    for p in path_list:

        with open(p, "r") as f:
            content = f.read()
        content = [x for x in re.split(r"\s+", content) if x != ""]

        lines_to_write = []

        for i in range(0, len(content), 5):
            try:
                if content[0] == '1':
                    lines_to_write.append('1')
                    continue
                lines_to_write.append(
                    f"{int(content[i])} "
                    f"{float(content[i + 1]):.6f} "
                    f"{float(content[i + 2]):.6f} "
                    f"{float(content[i + 3]):.6f} "
                    f"{float(content[i + 4]):.6f}"
                )
            except ValueError:
                empty_label.append([p, "Non-numeric label values found."])
                break

        print(lines_to_write)

        with open(p, "w") as f:
            f.write("\n".join(lines_to_write))

print("Done")

### 3.1.3 Change class ID

Change class ID for all the labels.
For exemple, it will replace all the 0 class by 1.

⚠️Be careful, be sure to carefully set the rights labels paths. It could replace the class in all labels files.

In [ ]:
current_class_id = 0 # int | None
new_class_id = 0
class_id_to_update = [
    dataset_structure["labels"],
]

try:
    if input("Files are going to be changed. Write yes to continue.") == 'yes':
        for path_list in class_id_to_update:
            for p in path_list:
                with open(p, "r") as f:
                    content = f.read()

                # split and REMOVE empty strings
                content = [x for x in re.split(r"\s+", content) if x]

                # safety check: must be divisible by 5
                if len(content) % 5 != 0:
                    print(f"Skipping malformed label file: {p}")
                    continue

                # update class IDs
                for i in range(0, len(content), 5):
                    if content[i] == str(current_class_id):
                        content[i] = str(new_class_id)

                if current_class_id is None and p.stat().st_size == 0:
                    with open(p, "w") as f:
                        f.write(str(new_class_id))
                    continue

                # rebuild content: one object per line
                rows = [
                    " ".join(content[i:i + 5])
                    for i in range(0, len(content), 5)
                ]

                content = "\n".join(rows) + "\n"

                # write back to file
                with open(p, "w") as f:
                    f.write(content)

except KeyboardInterrupt:
    print("Change aborted.")


### 3.1.4 Set labels (SAM 3)

This section aims to auto-label images by using [SAM 3](https://ai.meta.com/research/sam3/) developed by Meta. It can be used for any object recogintion (drone, birds...).

⚠️ Nvidia GPU is required for SAM3

In [ ]:
from sam3.model.sam3_image_processor import Sam3Processor
from sam3.visualization_utils import plot_results, normalize_bbox
from sam3 import build_sam3_image_model

Download bpe_simple_vocab_16e6.txt.gz on [GitHub](https://github.com/openai/CLIP/blob/main/clip/bpe_simple_vocab_16e6.txt.gz) in the models directory (./models).

In [ ]:
bpe_name = "bpe_simple_vocab_16e6.txt.gz"

##### Load model

In [ ]:
model = build_sam3_image_model(bpe_path=os.path.join(MODELS_DIRECTORY, bpe_name))
processor = Sam3Processor(model)

Once the model has been loaded successfully, SAM3 should be ready to use.
The section below is not for the dataset labeling. It won't write anything on your disk. It's only to test the model on image or video.

To write labels please go to the next section.

**Parameters:**
- `sam_sample_image_path`: Image/video path to give to SAM3.
- `prompt`: Object to tell for recognition. Eg: drone, bird...
- `confidence_threshold`: Give result above the given threshold.

In [ ]:
sam_sample_image_path = f"1.png"
prompt = ""
confidence_threshold = 0.2

image = Image.open(sam_sample_image_path)
width, height = image.size
processor = Sam3Processor(model, confidence_threshold=confidence_threshold)
inference_state = processor.set_image(image)

# --- Reset Prompts and Set Text Prompt ---
processor.reset_all_prompts(inference_state)
inference_state = processor.set_text_prompt(state=inference_state, prompt=prompt)
img0 = Image.open(sam_sample_image_path)
plot_results(img0, inference_state)

##### Settings

Both sections below aim to create a label if the specified object is detected on the image.

- **labelize_overwrite_label:** If True, overwrite the label if it already exists.
- **labelize_prompt:** Object to tell for recognition.
- **labelize_confidence_threshold:** Give a result above the given threshold.
- **labelize_object_class:** Class ID to give to the object.
- **labelize_output_label_file_directory:** Directory where to save the labels.
- **labelize_output_label_file_errors:** File where to save the errors.
- **labelize_output_label_file_success:** File where to save the success.

In [ ]:
labelize_overwrite_label: bool = True
labelize_prompt: str = 'drone'
labelize_confidence_threshold: float = 0.62
labelize_object_class = 0
labelize_output_label_file_directory = os.path.join(DATASET_ROOT_DIR, 'new_labels')
labelize_output_label_file_errors = os.path.join(labelize_output_label_file_directory, 'errors.txt')
labelize_output_label_file_success = os.path.join(labelize_output_label_file_directory, 'success.txt')

liberalizations_path = [
    dataset_structure["images"]
]

Start labeling... There is no configuration here.

In [ ]:
os.makedirs(labelize_output_label_file_directory, exist_ok=True)

image_failed_labelize = []

for paths in liberalizations_path:
    for image_path in paths:
        root, ext = os.path.splitext(image_path)
        root_label_name = root.split("/")[-1]
        current_label_path = root + ".txt"
        saved_label_path = os.path.join(labelize_output_label_file_directory, root_label_name) + ".txt"

        if os.path.isfile(current_label_path) and not labelize_overwrite_label:
            continue

        if os.path.isfile(saved_label_path) and not labelize_overwrite_label:
            continue

        # Load image
        image = PILImage.open(image_path)
        if image.mode != "RGB":
            with open(labelize_output_label_file_errors, "a") as f:
                image_failed_labelize.append(image_path)
                f.write(f"{image_path} is not RGB.\n")
            continue
        width, height = image.size

        # Run SAM3 processor
        processor = Sam3Processor(model, confidence_threshold=labelize_confidence_threshold)
        inference_state = processor.set_image(image)
        processor.reset_all_prompts(inference_state)
        inference_state = processor.set_text_prompt(state=inference_state, prompt=labelize_prompt)

        # --- Save YOLO labels ---
        bboxes = inference_state.get("boxes", None)  # fixed

        if bboxes is None or len(bboxes) == 0:
            # print(f"No detections for {image_path}")
            image_failed_labelize.append(image_path)
            with open(labelize_output_label_file_errors, "a") as f:
                f.write(f"No detections for {image_path}\n")
            continue

        yolo_lines = []
        for box in bboxes:
            x_min, y_min, x_max, y_max = box  # SAM3 gives XYXY

            # Convert to YOLO format
            bbox_width = x_max - x_min
            bbox_height = y_max - y_min

            cx = (x_min + bbox_width / 2) / width
            cy = (y_min + bbox_height / 2) / height

            nw = bbox_width / width
            nh = bbox_height / height

            yolo_lines.append(
                f"{labelize_object_class} {cx:.6f} {cy:.6f} {nw:.6f} {nh:.6f}"
            )

        with open(saved_label_path, "w") as f:
            f.write("\n".join(yolo_lines))
        with open(labelize_output_label_file_success, "a") as f:
            f.write(f"Saved YOLO labels: {saved_label_path}\n")

    with open(labelize_output_label_file_success, 'r') as fp:
        nb_lines = len(fp.readlines())
        print(f"Successfully procedded files: {nb_lines} out of {len(paths)}")

In [ ]:
 delete_files_in_dataset(image_failed_labelize)

## 🗃️  3.2 Files

### 3.1.2 Split dataset into folders

Split images and label into N folders.

**Parameters:**
- **data_file_per_folder**: Number of images/label pairs per folder
- **images_directory**: Directory containing the images and labels to split

In [ ]:
data_file_per_folder = 1000  # Couple of label and image
images_directory = Path('datasets/drone_detection/train_validation_test')

images_path = sort_files_by_number(
    list_files(images_directory, ALLOWED_EXTENSIONS, True)
)

nb_folders = math.ceil(len(images_path) / data_file_per_folder)

(images_directory / "output").mkdir()

for i in range(nb_folders):
    (images_directory / "output" / str(i)).mkdir()
    #
    for y in range(data_file_per_folder):
        index = i * data_file_per_folder + y
        if index >= len(images_path):
            break
        # Copy image
        shutil.copy(images_path[index], images_directory / "output" / str(i) / images_path[index].name)

        # Copy label
        label_path = images_path[index].with_suffix(".txt")
        if os.path.exists(label_path):
            shutil.copy(label_path, images_directory / "output" / str(i) / label_path.name)

### 3.1.3 Merge file from split folders

Reverse the process explain in the previous section

In [ ]:
input_folder_directory = Path("train_validation_test")
output_folder_directory = Path("train_validation_test_merged")

images_path = sort_files_by_number(
    list_files(input_folder_directory, ALLOWED_EXTENSIONS, True, True)
)

output_folder_directory.mkdir()

for i in range(len(images_path)):
    shutil.copy(images_path[i], output_folder_directory / images_path[i].name)

    # Copy label
    label_path = images_path[i].with_suffix(".txt")
    if os.path.exists(label_path):
        shutil.copy(label_path, output_folder_directory / label_path.name)

### 3.1.4  Rename files

Rename images and labels from `start_index` to n images.

`start_index` can be changed to merge dataset eaiser and avoid name conflict.

In [ ]:
start_index = 0


def numeric_key(name):
    nums = re.findall(r'\d+', name)
    return int(nums[0]) if nums else float('inf')


def collect_images(folder):
    return sorted(
        [f for f in os.listdir(folder) if f.endswith(tuple(ALLOWED_EXTENSIONS))],
        key=numeric_key
    )


def process(folder, start_i, out_dir):
    files = collect_images(folder)
    print(f"Processing {out_dir}...")

    if not os.path.exists(out_dir):
        os.mkdir(out_dir)

    for f in files:
        pass
        root, ext = os.path.splitext(f)

        # find real existing image
        image_path = None
        for e in ALLOWED_EXTENSIONS:
            p = os.path.join(folder, root + e)
            if os.path.isfile(p):
                image_path = p
                break

        if image_path is None:
            print("Missing image for:", f)
            continue

        label_path = os.path.join(folder, root + ".txt")
        if not os.path.isfile(label_path):
            os.rename(image_path, os.path.join(out_dir, f"{start_i}.jpg"))
            start_i += 1
            continue

        new_image = os.path.join(out_dir, f"{start_i}.jpg")
        new_label = os.path.join(out_dir, f"{start_i}.txt")

        # Move safely
        os.rename(image_path, new_image)
        os.rename(label_path, new_label)

        start_i += 1

    os.rmdir(folder)
    os.rename(out_dir, folder)

    return start_i


# === RUN ===
# backup_dataset()

tmp = os.path.join(DATASET_ROOT_DIR, "tmp")

start_index = process(DATASET_SPLIT_NAME, start_index, tmp + '_all')

update_dataset_structure()

print(f"DONE — All files renamed safely into {DATASET_ROOT_DIR}tmp_NAME")

### 3.1.5 Split Videos

Split videos into frames

In [ ]:
videos_directory = Path("/run/media/adrien/852ab715-3aaa-4da8-a8de-9f9524a00686/Datasets/AXAIR")

output_dir = videos_directory / "output"
output_dir.mkdir(exist_ok=True)

frame_rate = 5  # frames per second
frame_interval = int(1000 / frame_rate)  # in milliseconds

videos_files = list(videos_directory.glob("*.mp4"))

saved_count = 0
for video in videos_files:
    print(f"Processing: {video}")

    cap = cv2.VideoCapture(str(video))

    frame_count = 0

    while True:
        # Set position in milliseconds
        cap.set(cv2.CAP_PROP_POS_MSEC, frame_count * frame_interval)

        success, frame = cap.read()
        if not success:
            break

        output_path = output_dir / f"{saved_count}.jpg"
        cv2.imwrite(str(output_path), frame)

        saved_count += 1
        frame_count += 1

    cap.release()

print("Done.")

### 3.1.6 Split by label

Separate image and labels based on label

In [ ]:
labels_to_split = [
    dataset_structure["labels"],
]

try:
    if input("Files are going to be changed. Write yes to continue.") == 'yes':
        for path_list in labels_to_split:
            for p in path_list:
                with open(p, "r") as f:
                    content = f.read()

                # split and REMOVE empty strings
                content = [x for x in re.split(r"\s+", content) if x]

                if content[0] == '1' or content[0] == '0':
                    new_directory = p.parent / content[0]
                    Path(new_directory).mkdir(exist_ok=True)

                    file_without_extension = p.stem
                    for file in p.parent.glob(f"{file_without_extension}.*"):

                        shutil.move(file, new_directory / file.name)

except KeyboardInterrupt:
    print("Change aborted.")
